# Reeling in Smart Attackers' Phishing Emails — Colab notebook

**CSE 587 — Group 5**: Parker Davis · Dillon VanGilder · Emmanuel Adjei Domfeh.

End-to-end pipeline (multi-seed, includes `char_perturb`):
1. Install dependencies
2. Clone the project from GitHub
3. (Optional) Mount Drive to back up checkpoints
4. Download the Champa et al.\ curated phishing dataset
5. Inspect the dataset
6. **Train baseline BERT — 3 seeds**
7. **Train BERT with smart-attacker augmentation — 3 seeds**
8. **Aggregate per-seed reports → mean ± std**
9. Visualize headline + per-edit ablation
10. Inspect smart-attacker examples

Runtime: **A100 GPU**. Total wall time ≈ 50–60 min for 3 baseline + 3 augmented runs.

## 1. Install dependencies

In [ ]:
!pip install -q transformers==4.45.2 datasets==2.21.0 accelerate==0.34.2 \
    scikit-learn pandas matplotlib seaborn nltk requests pyarrow

## 2. Clone the project from GitHub

In [ ]:
!git clone https://github.com/eadomfeh1/cse587-phishing.git
%cd cse587-phishing

In [ ]:
import os, sys
sys.path.insert(0, os.getcwd())
print('CWD:', os.getcwd())
print('Files:', sorted(os.listdir('.')))

## 2b. (Optional) Back up results to Google Drive

Recommended: with 6 runs, you really want the checkpoints persisted somewhere if Colab disconnects.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
os.makedirs('/content/drive/MyDrive/cse587_results', exist_ok=True)

## 3. Download the dataset

In [ ]:
!python scripts/download_data.py
!ls -lh data/raw

## 4. Inspect the dataset

In [ ]:
from src.data_loader import load_raw_csvs
df = load_raw_csvs()
print('Total rows:', len(df))
print(df.label.value_counts())
df.head(3)

## 5. Train baseline BERT — 3 seeds

Each run writes to `results/baseline_seed{N}/` (checkpoint, eval_report.json, report.md). Splits are cached in `data/processed/`, so the second and third runs reuse them and start training immediately.

In [ ]:
for seed in (42, 123, 2024):
    print(f'\n===== BASELINE seed={seed} =====')
    !python -m src.train --model bert-base-uncased --epochs 3 --batch_size 64 --seed {seed}

In [ ]:
# Back up baseline runs to Drive (no-op if Drive isn't mounted)
!cp -r results /content/drive/MyDrive/cse587_results/baseline_3seeds_$(date +%Y%m%d_%H%M%S) 2>/dev/null && echo 'baseline runs saved to Drive' || echo 'Drive not mounted'

## 6. Train with smart-attacker augmentation — 3 seeds

Each run writes to `results/augmented_seed{N}/`. Same three seeds, with `--augment --augment_factor 1`. The default `aug_ops` now includes **`char_perturb`** (homoglyph + adjacent-letter swap) in addition to the previous four edits.

In [ ]:
for seed in (42, 123, 2024):
    print(f'\n===== AUGMENTED seed={seed} =====')
    !python -m src.train --model bert-base-uncased --epochs 3 --augment --augment_factor 1 --batch_size 64 --seed {seed}

In [ ]:
!cp -r results /content/drive/MyDrive/cse587_results/augmented_3seeds_$(date +%Y%m%d_%H%M%S) 2>/dev/null && echo 'augmented runs saved to Drive' || echo 'Drive not mounted'

## 7. Aggregate per-seed reports

Reads every `results/{baseline|augmented}_seed*/eval_report.json` and produces mean ± std for each metric, plus deltas, and a per-edit ablation table.

In [ ]:
!python scripts/aggregate_seeds.py

In [ ]:
# Display the markdown summary inline
from IPython.display import Markdown
Markdown(open('results/aggregated.md').read())

## 8. Visualize: headline + per-edit ablation

In [ ]:
import json, numpy as np, matplotlib.pyplot as plt
agg = json.loads(open('results/aggregated.json').read())
configs = agg['configs']

labels = ['Standard test', 'Smart-attacker test']
x = np.arange(len(labels))
width = 0.38

fig, ax = plt.subplots(figsize=(7, 4.2))
for i, name in enumerate(['baseline', 'augmented']):
    if name not in configs:
        continue
    c = configs[name]
    means = [c['standard']['f1_phish']['mean'], c['smart_attacker']['f1_phish']['mean']]
    stds  = [c['standard']['f1_phish']['std'],  c['smart_attacker']['f1_phish']['std']]
    ax.bar(x + (i - 0.5) * width, means, width, yerr=stds, capsize=4, label=f"{name} (n={c['n_seeds']})")

ax.set_xticks(x); ax.set_xticklabels(labels)
ax.set_ylabel('F1 (phishing)')
ax.set_ylim(0.98, 1.0)
ax.set_title('F1 (phishing): baseline vs augmented, 3 seeds (mean ± std)')
ax.legend()
plt.tight_layout(); plt.show()

In [ ]:
# Per-edit ablation, augmented model, mean ± std
abl = configs['augmented']['ablation']
ops = list(abl.keys())
means = [abl[o]['f1_phish']['mean'] for o in ops]
stds  = [abl[o]['f1_phish']['std']  for o in ops]

fig, ax = plt.subplots(figsize=(7, 4.2))
ax.bar(ops, means, yerr=stds, capsize=4, color='steelblue')
std_mean = configs['augmented']['standard']['f1_phish']['mean']
ax.axhline(std_mean, color='black', linestyle='--', label=f'standard F1 ({std_mean:.4f})')
ax.set_ylim(0.98, 1.0)
ax.set_ylabel('F1 (phishing)')
ax.set_title('Per-edit ablation, augmented model (3 seeds, mean ± std)')
plt.xticks(rotation=20); plt.legend(); plt.tight_layout(); plt.show()

## 9. Inspect smart-attacker examples (now includes char_perturb)

In [ ]:
from src.augmentation import AugmentationConfig, make_smart_attacker_eval
from src.data_loader import load_splits
splits = load_splits()
smart = make_smart_attacker_eval(splits['test'].head(20).copy(), AugmentationConfig())
for _, row in smart[smart.label == 1].head(3).iterrows():
    print('---')
    print(row['text'][:600])